# SI4006 · Sesión 14 · Lab: **Destilar un componente de su sistema**  ·  RESUELTO ✅

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 5

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manularrea/EAFIT-SI4006/blob/main/sesiones/s14/S14_Lab_Destilacion_RESUELTO.ipynb)

---

Hoy no agregan capacidades: hacen que una pieza sea **más chica y más rápida**, y miden cuánta calidad pagan. Dos partes:

- **Parte A (todos) · Destilar por logits:** un ViT-base (profesor) le enseña a un modelo chico (estudiante) a clasificar imágenes de su dominio, con etiquetas suaves y temperatura. Tabla antes/después: accuracy, latencia, tamaño.
- **Parte B (si su cuello de botella es el generador) · Datos sintéticos + LoRA:** el profesor (Qwen2.5-1.5B) escribe pares pregunta–respuesta de su dominio; un estudiante (Qwen2.5-0.5B) se fine-tunea con LoRA (la receta de S04). Se mide con su harness.

> **Traigan:** 30–100 imágenes etiquetadas de su dominio (o usen el dataset de ejemplo) y, para la parte B, su `responder_m3` o 20 preguntas gold de M2.
>
> GPU **T4**. La parte A corre en CPU (lento); la B necesita GPU.

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.45" datasets accelerate peft scikit-learn pillow
import torch, time, json, numpy as np
import torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

## 1 · Las imágenes y el contrato

Antes de entrenar nada, **escriban el contrato**: cuánta calidad aceptan perder a cambio de cuánta velocidad. Si no lo escriben antes, después cualquier resultado parece bueno.

In [ ]:
# 👉 TODO 1 · el contrato de eficiencia (se copia al README de M5)
CONTRATO = {'componente': 'clasificador de hojas (ViT)',          # ← cambien
            'acepto_perder_puntos': 3,                             # accuracy (o puntos del harness)
            'quiero_ganar': 'latencia ≥ 5× y caber en CPU'}
print(CONTRATO)

# Dataset de ejemplo: 'beans' (hojas de fríjol: sana / roya angular / mancha) — 3 clases, pequeño, de HF.
# 👉 Reemplacen por el suyo: un ImageFolder con sus imágenes etiquetadas (load_dataset('imagefolder', data_dir=...)).
from datasets import load_dataset
ds = load_dataset('beans')
train = ds['train'].shuffle(seed=7).select(range(400)); test = ds['test']
CLASES = ds['train'].features['labels'].names; print(CLASES, len(train), len(test))

## 2 · Parte A · El profesor: un ViT-base que ya sabe

No lo entrenamos desde cero: tomamos un ViT preentrenado y lo ajustamos **rápido** a nuestras clases (ruta B de S12: extractor + cabeza). Ese es el profesor. Lo importante no es que sea perfecto: es que sea **mejor y más pesado** que el estudiante.

In [ ]:
from transformers import ViTModel, ViTImageProcessor
PROF_ID = 'google/vit-base-patch16-224'
proc = ViTImageProcessor.from_pretrained(PROF_ID)
vit = ViTModel.from_pretrained(PROF_ID).to(device).eval()

@torch.no_grad()
def feats(imgs):
    ent = proc(images=imgs, return_tensors='pt').to(device)
    return vit(**ent).last_hidden_state[:, 0]          # el token CLS (S11)

def featurizar(split, bs=32):
    X = [];
    for i in range(0, len(split), bs): X.append(feats(split[i:i+bs]['image']).cpu())
    return torch.cat(X), torch.tensor(split['labels'])
Xtr, ytr = featurizar(train); Xte, yte = featurizar(test); print(Xtr.shape, Xte.shape)

In [ ]:
# la cabeza del profesor: una capa lineal sobre el CLS (entrena en segundos)
cabeza = nn.Linear(768, len(CLASES)).to(device)
opt = torch.optim.Adam(cabeza.parameters(), lr=1e-3)
for ep in range(30):
    idx = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        b = idx[i:i+64]; loss = F.cross_entropy(cabeza(Xtr[b].to(device)), ytr[b].to(device)); opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad(): acc_prof = (cabeza(Xte.to(device)).argmax(1).cpu() == yte).float().mean().item()
print(f'PROFESOR · accuracy test: {acc_prof:.3f}')

In [ ]:
def profesor(imgs):
    """imagen(es) → logits del profesor. Esto es lo que el estudiante va a imitar."""
    with torch.no_grad(): return cabeza(feats(imgs)).cpu()

def cronometrar(f, imgs, n=3):
    f(imgs[:1]); t0 = time.time()
    for _ in range(n): f(imgs)
    return (time.time() - t0) / n / len(imgs)
lote = test[:16]['image']
lat_prof = cronometrar(profesor, lote)
n_prof = sum(p.numel() for p in vit.parameters()) + sum(p.numel() for p in cabeza.parameters())
print(f'profesor: {n_prof/1e6:.1f} M parámetros · {lat_prof*1000:.1f} ms/imagen ({device})')

## 3 · Parte A · Etiquetas suaves y temperatura

Miren la diferencia entre lo que dice el dataset (dura) y lo que dice el profesor (suave), y cómo la temperatura la "ablanda".

In [ ]:
z = profesor(test[:3]['image'])
for i in range(3):
    print(f'imagen {i} · etiqueta dura: {CLASES[yte[i]]}')
    for T in (1, 2, 4):
        p = F.softmax(z[i] / T, dim=0)
        print(f'   T={T}:  ' + '  '.join(f'{c}={p[j]:.2f}' for j, c in enumerate(CLASES)))
# 👀 con T=1 casi es una etiqueta dura; con T=4 aparecen los parecidos entre clases. Eso es lo que vale copiar.

## 4 · Parte A · El estudiante

Un modelo **chico**: una CNN pequeña que trabaja sobre la imagen a 64×64 (nada de ViT, nada de 86 M parámetros). Lo entrenamos dos veces, con las mismas imágenes:

- **sin profesor:** solo con las etiquetas duras (cross-entropy),
- **con profesor (destilado):** imitando los logits suaves del profesor con temperatura, más un poco de etiqueta dura.

Si la destilación sirve, el segundo debe ganarle al primero **con el mismo tamaño**.

In [ ]:
from torchvision import transforms
tf = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
def tensorizar(split): return torch.stack([tf(im.convert('RGB')) for im in split['image']])
Itr, Ite = tensorizar(train), tensorizar(test)
Ztr = profesor(train['image'])            # logits del profesor sobre TODO el train (una vez)

class Estudiante(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.h = nn.Linear(64, n)
    def forward(self, x): return self.h(self.f(x))
print('estudiante:', sum(p.numel() for p in Estudiante(len(CLASES)).parameters()) / 1e3, 'K parámetros')

In [ ]:
def perdida_destilacion(z_est, z_prof, y, T=2.0, alpha=0.7):
    """alpha · KL(softmax(z_prof/T) || softmax(z_est/T)) · T² + (1-alpha) · CE(y, z_est)"""
    # 👉 TODO 2 · implementen la fórmula de la slide. Pistas: F.log_softmax, F.softmax, F.kl_div(..., reduction='batchmean'), F.cross_entropy.
    #    El factor T² mantiene la escala de los gradientes (Hinton, 2015).
    kl = F.kl_div(F.log_softmax(z_est / T, dim=1), F.softmax(z_prof / T, dim=1), reduction='batchmean') * (T ** 2)
    return alpha * kl + (1 - alpha) * F.cross_entropy(z_est, y)

def entrenar(destilar, T=2.0, alpha=0.7, epocas=25, seed=0):
    torch.manual_seed(seed); m = Estudiante(len(CLASES)).to(device); opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    for ep in range(epocas):
        idx = torch.randperm(len(Itr))
        for i in range(0, len(Itr), 32):
            b = idx[i:i+32]; x, y, zp = Itr[b].to(device), ytr[b].to(device), Ztr[b].to(device)
            loss = perdida_destilacion(m(x), zp, y, T, alpha) if destilar else F.cross_entropy(m(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad(): acc = (m(Ite.to(device)).argmax(1).cpu() == yte).float().mean().item()
    return m, acc

est_solo, acc_solo = entrenar(destilar=False); print(f'estudiante SIN profesor · accuracy: {acc_solo:.3f}')
est_dest, acc_dest = entrenar(destilar=True, T=2.0);  print(f'estudiante DESTILADO (T=2) · accuracy: {acc_dest:.3f}')

In [ ]:
# 👉 TODO 3 · barran la temperatura con la MISMA semilla y anoten: ¿cuál T ayuda más? ¿alguna empeora?
for T in (1.0, 2.0, 4.0, 8.0):
    _, a = entrenar(destilar=True, T=T); print(f'T={T}: accuracy {a:.3f}')

## 5 · Parte A · La tabla antes/después

La que va al README de M5 y a la demo. Calidad, latencia y tamaño: profesor vs. estudiante, **en la misma máquina**.

In [ ]:
def estudiante_fn(imgs):
    with torch.no_grad(): return est_dest(torch.stack([tf(im.convert('RGB')) for im in imgs]).to(device)).cpu()
lat_est = cronometrar(estudiante_fn, lote)
n_est = sum(p.numel() for p in est_dest.parameters())
import pandas as pd
tabla = pd.DataFrame([
    {'modelo': 'profesor (ViT-base + cabeza)', 'accuracy': round(acc_prof, 3), 'ms/imagen': round(lat_prof*1000, 1), 'parámetros': f'{n_prof/1e6:.1f} M'},
    {'modelo': 'estudiante sin profesor',       'accuracy': round(acc_solo, 3), 'ms/imagen': round(lat_est*1000, 1),  'parámetros': f'{n_est/1e3:.0f} K'},
    {'modelo': 'estudiante DESTILADO',          'accuracy': round(acc_dest, 3), 'ms/imagen': round(lat_est*1000, 1),  'parámetros': f'{n_est/1e3:.0f} K'},
])
display(tabla)
perdio = (acc_prof - acc_dest) * 100
print(f"Contrato: aceptaban perder {CONTRATO['acepto_perder_puntos']} puntos · perdieron {perdio:.1f} · ganaron {lat_prof/lat_est:.0f}× en latencia y {n_prof/n_est:.0f}× en tamaño → "
      + ('CUMPLIDO ✅' if perdio <= CONTRATO['acepto_perder_puntos'] else 'NO cumplido ❌ (suban T, más datos, o un estudiante más grande)'))

In [ ]:
# 👀 el caso donde el estudiante falla y el profesor no: ahí está lo que la destilación no pudo copiar
with torch.no_grad():
    pp = profesor(test['image']).argmax(1); pe = estudiante_fn(test['image']).argmax(1)
mal = [(i, CLASES[yte[i]], CLASES[pe[i]]) for i in range(len(yte)) if pp[i] == yte[i] and pe[i] != yte[i]]
print(len(mal), 'casos donde el profesor acierta y el estudiante no. Los primeros:', mal[:5])
# 👉 TODO 4 · miren 3 de esas imágenes: ¿son casos raros? ¿clases parecidas? Anótenlo para el README.

## 6 · Parte B (opcional, GPU) · Datos sintéticos + LoRA para el generador

Si su cuello de botella es el **generador** (el agente llama al LLM 3–5 veces por pregunta), la forma B: el profesor escribe el dataset, el estudiante se fine-tunea con LoRA (S04). Se mide con **su harness de M2**, no con accuracy.

> Licencia: Qwen2.5 permite usar sus salidas para entrenar. Con APIs cerradas (OpenAI, Anthropic, Google) **no**: léanla antes.

In [ ]:
# B1 · el profesor escribe pares (pregunta, respuesta) de SU dominio
# from transformers import AutoModelForCausalLM, AutoTokenizer
# PROF = 'Qwen/Qwen2.5-1.5B-Instruct'; tok = AutoTokenizer.from_pretrained(PROF)
# prof = AutoModelForCausalLM.from_pretrained(PROF, torch_dtype=torch.float16, device_map='auto')
# TEMAS = ['roya del café', 'minador de la hoja', 'riego en época seca']            # ← 20–50 temas de su dominio
# def generar(prompt, n=160):
#     ent = tok.apply_chat_template([{'role': 'user', 'content': prompt}], add_generation_prompt=True, return_tensors='pt').to(prof.device)
#     out = prof.generate(ent, max_new_tokens=n, do_sample=True, temperature=0.7); return tok.decode(out[0][ent.shape[1]:], skip_special_tokens=True)
# pares = []
# for t in TEMAS:
#     for _ in range(10):
#         q = generar(f'Escribe UNA pregunta corta que haría un caficultor sobre: {t}. Solo la pregunta.')
#         a = generar(f'Responde en 3 frases, en español, como un agrónomo: {q}')
#         pares.append({'pregunta': q.strip(), 'respuesta': a.strip()})
# json.dump(pares, open('pares_sinteticos.json', 'w'), ensure_ascii=False, indent=1); print(len(pares), 'pares')

In [ ]:
# B2 · el estudiante (Qwen2.5-0.5B) se fine-tunea con LoRA sobre esos pares: la receta de S04, sin cambios
# from peft import LoraConfig, get_peft_model
# EST = 'Qwen/Qwen2.5-0.5B-Instruct'; etok = AutoTokenizer.from_pretrained(EST)
# est = AutoModelForCausalLM.from_pretrained(EST, torch_dtype=torch.float16, device_map='auto')
# est = get_peft_model(est, LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM'))
# ... (bucle de entrenamiento de S04 sobre `pares`, 2–3 épocas) ...
# B3 · medir con SU harness: harness(eval_set, sistema_con_estudiante) vs. harness(eval_set, sistema_con_profesor) + latencia p50/p95
# → la misma tabla antes/después de la parte A, con las columnas de su scorecard en vez de accuracy.

## Lo que se llevan (y lo que va a M5)

- **Destilar** = entrenar un modelo chico a imitar las salidas (suaves, con temperatura) de uno grande. Con el mismo tamaño, el destilado le gana al entrenado solo con etiquetas.
- **El contrato se escribe antes**: cuánta calidad aceptan perder por cuánta velocidad. La tabla antes/después dice si se cumplió.
- **Lo que no copia**: casos raros, la válvula de escape, y hereda sesgos. Los casos adversariales del eval set lo detectan.
- Para **M5**: una técnica de eficiencia sobre un componente, con esta tabla (o la de cuantización de S15), el contrato y la licencia del profesor en el README.

*SI4006 · Universidad EAFIT · Sesión 14 · RESUELTO.*